# SmartParkingGU - Conectar GPU de Colab a VS Code Local

Este notebook configura un servidor Jupyter en Google Colab y lo expone via tunnel para conectar desde VS Code.

## Instrucciones:
1. Ejecuta todas las celdas en orden
2. Copia la URL del tunnel que aparece en la CELDA 4
3. En VS Code: `Ctrl+Shift+P` > `Jupyter: Specify Jupyter Server URL`
4. Pega la URL y usa el token indicado

**Requisitos:** Colab con GPU (Runtime > Change runtime type > T4 GPU)

In [ ]:
# CELDA 1: Clonar repositorio e instalar dependencias
!git clone https://github.com/KaraIbr/SmartParkingGU.git
%cd SmartParkingGU
!pip install -r requirements.txt
!pip install jupyter notebook localtunnel

print("\n Dependencias instaladas")

In [ ]:
# CELDA 2: Verificar GPU
import torch
import subprocess

print("=" * 50)
print("VERIFICACION DE GPU")
print("=" * 50)
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA disponible: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    gpu_mem = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f"VRAM: {gpu_mem:.1f} GB")
    print("\n GPU lista para entrenar")
else:
    print("\n No hay GPU disponible")
    print("Ve a Runtime > Change runtime type > T4 GPU")
    print("Luego re-ejecuta esta celda")

In [ ]:
# CELDA 3: Verificar estructura del proyecto
import os

print("Estructura del proyecto:")
print("-" * 40)
for root, dirs, files in os.walk('.'):
    if '.git' not in root and '__pycache__' not in root:
        level = root.replace('.', '').count(os.sep)
        indent = '  ' * level
        print(f"{indent}{os.path.basename(root)}/")
        if level < 2:
            subindent = '  ' * (level + 1)
            for f in files[:3]:
                print(f"{subindent}{f}")
            if len(files) > 3:
                print(f"{subindent}... (+{len(files)-3} mas)")

In [ ]:
# CELDA 4: Iniciar servidor Jupyter + Tunnel
# Esta celda tarda ~30 segundos en generar la URL

import subprocess
import time
import threading

TOKEN = "smartparking"
PORT = 8888

# Iniciar Jupyter en background
print("Iniciando servidor Jupyter...")
jupyter_cmd = [
    "jupyter", "notebook",
    f"--port={PORT}",
    "--no-browser",
    "--ip=0.0.0.0",
    f"--NotebookApp.token={TOKEN}",
    "--allow-root",
    "--NotebookApp.allow_origin='*'"
]

jupyter_proc = subprocess.Popen(
    jupyter_cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT
)

time.sleep(3)
print("Servidor Jupyter iniciado")

# Instalar y ejecutar localtunnel
print("\nConfigurando tunnel...")
!npm install -g localtunnel 2>/dev/null

# Obtener URL del tunnel
tunnel_result = subprocess.run(
    ["lt", "--port", str(PORT)],
    capture_output=True,
    text=True,
    timeout=30
)
tunnel_url = tunnel_result.stdout.strip()

print("\n" + "=" * 60)
print("URL DEL TUNNEL (copiar para VS Code)")
print("=" * 60)
print(f"\n  {tunnel_url}\n")
print("=" * 60)
print(f"Token: {TOKEN}")
print("=" * 60)
print("\nPasos en VS Code:")
print("  1. Ctrl+Shift+P")
print("  2. Buscar: Jupyter: Specify Jupyter Server URL")
print("  3. Pegar la URL de arriba")
print("  4. Ingresar token: smartparking")
print("  5. Seleccionar el kernel de Python")
print("\n*** MANTENESTE NOTEBOOK ABIERTO EN COLAB ***")

In [ ]:
# CELDA 5: Ejecutar notebooks del proyecto
# Descomenta el que quieras ejecutar

# Opcion A: Ejecutar desde terminal (sin interfaz)
# !jupyter nbconvert --to notebook --execute notebooks/02_base_model_cnrpark.ipynb --output ejecutado_02.ipynb
# !jupyter nbconvert --to notebook --execute notebooks/03_finetune_cnrpark.ipynb --output ejecutado_03.ipynb

# Opcion B: Ejecutar directamente
import subprocess
import sys

# Ejecutar notebook de modelo base
print("Ejecutando notebook 02_base_model_cnrpark.ipynb...")
result = subprocess.run(
    [
        sys.executable, "-m", "jupyter", "nbconvert",
        "--to", "notebook",
        "--execute",
        "notebooks/02_base_model_cnrpark.ipynb",
        "--output", "ejecutado_02.ipynb"
    ],
    capture_output=True,
    text=True
)

if result.returncode == 0:
    print(" Notebook 02 completado")
else:
    print(f" Error en notebook 02: {result.stderr[:500]}")

In [ ]:
# CELDA 6: Ver resultados
import pandas as pd

results_path = 'experiments/cnrpark/results.csv'
if os.path.exists(results_path):
    results = pd.read_csv(results_path)
    print("Resultados de experimentos:")
    print("=" * 60)
    cols = ['experiment_id', 'learning_rate', 'weight_decay', 'accuracy', 'f1_score']
    available_cols = [c for c in cols if c in results.columns]
    print(results.nlargest(10, 'f1_score')[available_cols].to_string(index=False))
else:
    print("No hay resultados aun. Ejecuta el notebook 02 primero.")

In [ ]:
# CELDA 7: Descargar resultados
from google.colab import files
import zipfile

with zipfile.ZipFile('smartparking_results.zip', 'w') as zipf:
    if os.path.exists('experiments/cnrpark/results.csv'):
        zipf.write('experiments/cnrpark/results.csv')
    if os.path.exists('experiments/cnrpark/best_model/best_model.pt'):
        zipf.write('experiments/cnrpark/best_model/best_model.pt')
    for root, dirs, files_list in os.walk('reports/figures'):
        for file in files_list:
            zipf.write(os.path.join(root, file))

files.download('smartparking_results.zip')
print("Resultados descargados")

In [ ]:
# CELDA 8: Estado del tunnel y servidor
# Verificar que todo sigue corriendo

import subprocess

# Verificar Jupyter
jupyter_check = subprocess.run(
    ["pgrep", "-f", "jupyter"],
    capture_output=True,
    text=True
)
if jupyter_check.stdout.strip():
    print("Servidor Jupyter: ACTIVO")
else:
    print("Servidor Jupyter: INACTIVO - reinicia la CELDA 4")

# Verificar GPU restante
import torch
if torch.cuda.is_available():
    allocated = torch.cuda.memory_allocated(0) / 1e9
    cached = torch.cuda.memory_reserved(0) / 1e9
    total = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f"\nGPU VRAM:")
    print(f"  Total:     {total:.1f} GB")
    print(f"  Asignada:  {allocated:.2f} GB")
    print(f"  Cache:     {cached:.2f} GB")
    print(f"  Libre:     {total - allocated:.2f} GB")